In [ ]:
#i will import important lybaries if i needed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Task 1: Read the dataset
Q1_data_path = os.path.join(path, 'Q1_data.csv')
df_Q1_data = pd.read_csv(Q1_data_path)


In [ ]:
# Task 2: Write your code here:
print(f"Shape: {df_Q1_data.shape}")
df_Q1_data.head()

In [ ]:
# Task 3: Write your code here:
df_Q1_data.info()

In [ ]:
# Task 4: Write your code here:
print("\nDataFrame Description:")
df.describe()

In [ ]:
# Task 5: Write your code here:
# delivery_time distribution
plt.figure(figsize=(10, 5))
plt.hist(df_Q1_data['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('delivery_time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=['Order_ID'])
print("Order_ID column dropped.")


In [ ]:
# Task 2: Write your code here:
#step -1 :i will see vigully the missing values
missing_info = df.isnull().sum()
print("Columns with missing values:\n", missing_info[missing_info > 0])
#step 2 : i wll put median for numerical data



In [ ]:
# Task 3: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")


In [ ]:
# Task 4: Write your code here:
# We use pd.get_dummies for One Hot Encoding
df = pd.get_dummies(df, drop_first=True) # drop_first=True avoids multicollinearity
print("Data encoded.")
df.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

# Separate Target from features first to avoid scaling the target
X = df.drop(columns=['Delivery_Time'])
y = df['Delivery_Time']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print("Features scaled.")
X_scaled.head()

In [ ]:
# Task 6: Write your code here:
skewness = y.skew()
print(f"Delivery_Time Skewness: {skewness:.2f}")

if abs(skewness) > 1:
    print("The Delivery_Time distribution is skewed (imbalanced).")
else:
    print("The Delivery_Time distribution is fairly symmetric.")

In [ ]:
# Task 1: Write your code here:
# Task 1: Split is already done above (X_scaled and y)
# Task 2: Import KFold
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Initialize KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []



In [ ]:
# Task 2,3,4,5: Write your code here:
# Task 3 & 4: Train Loop
print("Starting 5-Fold Cross-Validation...\n")

for fold, (train_index, val_index) in enumerate(kf.split(X_scaled, y)):
  # Split data
    X_train, X_val = X_scaled.iloc[train_index], X_scaled.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    # Initialize Model
    model = RandomForestRegressor(n_estimators=100, random_state=42)

    # Train line
    model.fit(X_train, y_train)

    # Predict line
    y_pred = model.predict(X_val)

    # Evaluate (MAE Only as the q)
    mae = mean_absolute_error(y_val, y_pred)
    mae_scores.append(mae)

    print(f"Fold {fold+1} MAE: {mae:.4f}")
    # Task 5: Average Score
print(f"\nAverage MAE: {np.mean(mae_scores):.4f}")



In [ ]:
# Task 1: Write your code here:
importance = pd.DataFrame({
    'feature': df_Q1_data_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feadf_Q1_datature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('df_Q1_data Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
#please chick the syntax, the code is not running
plt.figure(figsize=(8, 5))
sns.histplot(y_pred, kde=True, color='green', label='Predicted')
plt.title('Distribution of Predicted Delivery Time (Last Fold)')
plt.xlabel('Delivery Time (min)')
plt.legend()
plt.show()
#this will show how our model pretictid the delevary time how good it is

In [ ]:
# Task Bonus: Write your code here:
from catboost import CatBoostRegressor

# Re-initialize KFold (good practice to keep it clean)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
ensemble_mae_scores = []

print("Starting Ensemble Cross-Validation...\n")

for fold, (train_index, val_index) in enumerate(kf.split(X_scaled, y)):
  X_train, X_val = X_scaled.iloc[train_index], X_scaled.iloc[val_index]
  y_train, y_val = y.iloc[train_index], y.iloc[val_index]

  rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
  cb_model = CatBoostRegressor(verbose=0, random_state=42)


  rf_model.fit(X_train, y_train)
  cb_model.fit(X_train, y_train)


  rf_pred = rf_model.predict(X_val)
  cb_pred = cb_model.predict(X_val)


avg_pred = (rf_pred + cb_pred) / 2


  mae = mean_absolute_error(y_val, avg_pred)
  ensemble_mae_scores.append(mae)

print(f"Fold {fold+1} Ensemble MAE: {mae:.4f}")

print(f"\nAverage Ensemble MAE: {np.mean(ensemble_mae_scores):.4f}")